# pyVideoTrans – Notebook Edition

Translate a video from one language to another: speech recognition (ASR) → subtitle translation → text-to-speech (TTS) → final assembly.

This notebook wraps the [pyVideoTrans](https://github.com/jianchang512/pyvideotrans) backend without any GUI dependency.

---

## Pipeline overview

```
Input video
    ↓  [Cell 3] ASR  – extract audio → transcribe → source SRT
    ↓  [Cell 4] Translate – source SRT → target SRT
    ↓  [Cell 5] TTS  – target SRT → dubbed audio
    ↓  [Cell 6] Assemble – merge dubbed audio + video → output MP4
```

Each step can also be run independently.

## Cell 1 – Install dependencies

Run once. Skip on subsequent runs if the environment is already set up.

In [ ]:
import subprocess, sys

# Install core requirements (no GUI / Qt)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "faster-whisper",
    "openai-whisper",   # needed for RECOGN_TYPE=1
    "edge-tts",
    "srt",
    "pydub",
    "soundfile",
    "requests",
    "openai",
    "deepl",              # TRANSLATE_TYPE=16
    "deep-translator",
    "tqdm",
    "psutil",
    "ffmpeg-python",
    "nest_asyncio",
])
print("Dependencies installed.")

## Cell 2 – Configuration

Set all parameters here before running the pipeline cells.

In [ ]:
import os, re, sys
from pathlib import Path

# ─── Input / Output ───────────────────────────────────────────────────────────
INPUT_VIDEO   = "/path/to/your/video.mp4"   # absolute path to the source video
OUTPUT_DIR    = "/path/to/output"            # folder where results are saved

# ─── Languages ────────────────────────────────────────────────────────────────
SOURCE_LANG   = "zh"      # language code of the video's spoken language (e.g. zh, en, ja)
TARGET_LANG   = "en"      # language code for the translation output

# ─── ASR (Speech Recognition) ─────────────────────────────────────────────────
#   0  = Faster-Whisper  (local, recommended)
#   1  = OpenAI-Whisper  (local)
#   5  = OpenAI Speech API (remote, needs OPENAI_API_KEY)
#   6  = Gemini AI       (remote, needs GEMINI_API_KEY)
RECOGN_TYPE   = 0
WHISPER_MODEL = "large-v3-turbo"  # tiny | small | base | medium | large-v3-turbo | large-v3
USE_CUDA      = False             # set True if you have an NVIDIA GPU with CUDA

# ─── ASR quality / dubbing alignment (VTV and step cells) ───────────────────────
# REMOVE_NOISE: preprocess audio before ASR (noisy environments, phone mic).
REMOVE_NOISE   = False
# FIX_PUNC: restore punctuation after transcription (clearer text / TTS).
FIX_PUNC       = True
# With VOICE_AUTORATE or VIDEO_AUTORATE enabled, the aligner extends each cue's
# end time to the next cue's start (the gap belongs to the slot). See the
# docstring at the top of videotrans/task/_rate.py.
VOICE_AUTORATE = True   # speed up dubbed audio to fit the time slot
VIDEO_AUTORATE = True   # slow down video in the slot if TTS is still too long

# ─── Translation ──────────────────────────────────────────────────────────────
#   0  = Google Translate (free, no key needed)
#   1  = Microsoft Translate
#   3  = OpenAI ChatGPT   (needs OPENAI_API_KEY)
#   4  = DeepSeek         (needs DEEPSEEK_API_KEY)
#   5  = Gemini AI        (needs GEMINI_API_KEY)
#  16  = DeepL            (needs DEEPL_API_KEY)
TRANSLATE_TYPE = 0


# ─── Translation quality ──────────────────────────────────────────────────────
# When using an AI provider (DeepSeek, OpenAI, Gemini…) set True to send the
# entire SRT with timestamps — context dramatically improves quality.
# Must be False for Google / Microsoft.
AI_SENDS_SRT   = TRANSLATE_TYPE in [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 23, 24]

# Number of COMPLETE SENTENCES per batch (non-AI providers only).
# SRT line fragments are first merged into complete sentences (AutoDub style),
# then sentences are batched before each API call.
# Recommended: 10-20 (sentences, not SRT lines).
TRANS_THREAD   = 15

# Lines per batch when using AI with AI_SENDS_SRT=True.
AITRANS_THREAD = 30

# Send full video text as global context to the AI (better consistency, more tokens).
AI_GLOBAL_CONTEXT = True

# ─── TTS (Text-to-Speech) ─────────────────────────────────────────────────────
#   0  = Edge-TTS  (free, Microsoft, recommended)
#   15 = OpenAI TTS (remote, needs OPENAI_API_KEY)
#   19 = Gemini TTS (remote, needs GEMINI_API_KEY)
#  29  = Google TTS (free, gTTS)
TTS_TYPE      = 0
# Voice role for Edge-TTS: run list_edge_tts_voices() below to find available names
# Examples: "en-US-JennyNeural", "it-IT-ElsaNeural", "zh-CN-XiaoxiaoNeural"
VOICE_ROLE    = "en-US-JennyNeural"
VOICE_RATE    = "+0%"    # speed: e.g. "+20%" faster, "-20%" slower
VOLUME        = "+0%"    # volume adjustment
PITCH         = "+0Hz"   # pitch adjustment

# ─── Subtitle embedding in final video ────────────────────────────────────────
#   0 = no subtitles, 1 = hard (burned-in), 2 = soft (selectable)
SUBTITLE_TYPE = 1

# ─── Optional API keys (set as env vars or directly here) ────────────────────
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY", "")
GEMINI_API_KEY  = os.environ.get("GEMINI_API_KEY", "")
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
DEEPL_API_KEY   = os.environ.get("DEEPL_API_KEY", "")

# ─── Derived paths (do not edit) ──────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
_stem = Path(INPUT_VIDEO).stem
SOURCE_SRT  = str(Path(OUTPUT_DIR) / f"{SOURCE_LANG}.srt")  # matches trans_create.py naming
TARGET_SRT  = str(Path(OUTPUT_DIR) / f"{TARGET_LANG}.srt")  # matches trans_create.py naming
DUBBED_WAV  = str(Path(OUTPUT_DIR) / f"{_stem}_dubbed.wav")
OUTPUT_MP4  = str(Path(OUTPUT_DIR) / f"{_stem}_translated.mp4")

print(f"Input  : {INPUT_VIDEO}")
print(f"Output : {OUTPUT_DIR}")
print(f"Source : {SOURCE_LANG}  →  Target : {TARGET_LANG}")
print(f"ASR    : type={RECOGN_TYPE}, model={WHISPER_MODEL}, remove_noise={REMOVE_NOISE}, fix_punc={FIX_PUNC}")
print(f"Trans  : type={TRANSLATE_TYPE}")
print(f"TTS    : type={TTS_TYPE}, voice={VOICE_ROLE}")
print(f"Align  : voice_autorate={VOICE_AUTORATE}, video_autorate={VIDEO_AUTORATE}")

## Cell 3 – Initialise pyVideoTrans backend

In [ ]:
# Must be run before any videotrans import so paths and env vars are set correctly.
import sys, os
# Make sure the workspace root is on the Python path
WORKSPACE = str(Path(".").resolve())
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

from videotrans.configure import config
config.init_run()

from videotrans.configure.config import (
    ROOT_DIR, TEMP_DIR, app_cfg, settings, params, logger
)

# Expose API keys to the params store so individual channels can pick them up
if OPENAI_API_KEY:
    params['openaitts_key']      = OPENAI_API_KEY
    params['chatgpt_key']        = OPENAI_API_KEY
    params['openairecognapi_key']= OPENAI_API_KEY
if GEMINI_API_KEY:
    params['gemini_key']         = GEMINI_API_KEY
if DEEPSEEK_API_KEY:
    params['deepseek_key']       = DEEPSEEK_API_KEY
if DEEPL_API_KEY:
    params['deepl_key']          = DEEPL_API_KEY

app_cfg.exit_soft = False
app_cfg.exec_mode = 'cli'
# Applica le impostazioni di qualità traduzione
settings['trans_thread']     = TRANS_THREAD
settings['aitrans_thread']   = AITRANS_THREAD
settings['aisendsrt']        = AI_SENDS_SRT
settings['aitrans_context']  = AI_GLOBAL_CONTEXT


from videotrans.util import tools
from videotrans.util.gpus import getset_gpu
getset_gpu()

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"TEMP_DIR : {TEMP_DIR}")
print("Backend initialised successfully.")

## Cell 4 – ASR: Transcribe video to source-language SRT

Extracts audio from the video and runs speech recognition to produce an SRT subtitle file in the source language.

In [ ]:
import re, uuid
from pathlib import Path
from videotrans.task._speech2text import SpeechToText
from videotrans.task.taskcfg import TaskCfgSTT

# Prepare cache folder and file metadata
task_uuid   = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(INPUT_VIDEO).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

stt_cfg = TaskCfgSTT(
    **file_obj,
    uuid          = task_uuid,
    cache_folder  = cache_folder,
    target_dir    = OUTPUT_DIR,
    # ASR settings
    recogn_type   = RECOGN_TYPE,
    detect_language = SOURCE_LANG,
    model_name    = WHISPER_MODEL,
    is_cuda       = USE_CUDA,
    remove_noise  = REMOVE_NOISE,
    enable_diariz = False,
    rephrase      = 0,
    fix_punc      = FIX_PUNC,
)

print(f"Starting ASR with type={RECOGN_TYPE}, model={WHISPER_MODEL} …")
trk = SpeechToText(cfg=stt_cfg, out_format='srt')
trk.prepare()
trk.recogn()
trk.diariz()
trk.task_done()

# The output SRT is written by the task; copy it to our target path
import shutil
_srt_candidates = list(Path(cache_folder).glob("*.srt")) + list(Path(OUTPUT_DIR).glob(f"{file_obj['noextname']}*.srt"))
if _srt_candidates:
    shutil.copy(str(_srt_candidates[0]), SOURCE_SRT)
    print(f"Source SRT saved to: {SOURCE_SRT}")
    # Preview first 5 subtitles
    import srt
    subs = list(srt.parse(Path(SOURCE_SRT).read_text(encoding='utf-8')))
    print(f"\nTotal subtitles: {len(subs)}")
    for s in subs[:5]:
        print(f"  [{s.index}] {s.start} --> {s.end}")
        print(f"       {s.content.strip()}")
else:
    print("WARNING: No SRT file found. Check log for errors.")

## Cell 5 – Translate SRT to target language

Takes the source SRT and translates each subtitle line.

In [ ]:
from videotrans.task._translate_srt import TranslateSrt
from videotrans.task.taskcfg import TaskCfgSTS

sts_cfg = TaskCfgSTS(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = OUTPUT_DIR,
    source_sub           = SOURCE_SRT,
    source_language_code = SOURCE_LANG,
    target_language_code = TARGET_LANG,
    translate_type       = TRANSLATE_TYPE,
)

print(f"Translating from '{SOURCE_LANG}' to '{TARGET_LANG}' using provider {TRANSLATE_TYPE} …")
trk = TranslateSrt(cfg=sts_cfg, out_format=0)
trk.trans()
trk.task_done()

# Locate translated SRT
_trans_candidates = list(Path(cache_folder).glob("*target*.srt")) + list(Path(OUTPUT_DIR).glob(f"*{TARGET_LANG}*.srt"))
if _trans_candidates:
    import shutil
    shutil.copy(str(_trans_candidates[0]), TARGET_SRT)
    print(f"Target SRT saved to: {TARGET_SRT}")
    import srt
    subs = list(srt.parse(Path(TARGET_SRT).read_text(encoding='utf-8')))
    print(f"\nTotal subtitles: {len(subs)}")
    for s in subs[:5]:
        print(f"  [{s.index}] {s.content.strip()}")
elif sts_cfg.target_sub and Path(sts_cfg.target_sub).exists():
    import shutil
    shutil.copy(sts_cfg.target_sub, TARGET_SRT)
    print(f"Target SRT saved to: {TARGET_SRT}")
else:
    print("WARNING: Translated SRT not found. Check log for errors.")

## Cell 6 – TTS: Generate dubbed audio from target SRT

Synthesises speech for every subtitle line and stitches them into a single dubbed audio file.

In [ ]:
from videotrans.task._dubbing import DubbingSrt
from videotrans.task.taskcfg import TaskCfgTTS

tts_cfg = TaskCfgTTS(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = OUTPUT_DIR,
    target_sub           = TARGET_SRT,
    target_language_code = TARGET_LANG,
    # TTS settings
    tts_type             = TTS_TYPE,
    voice_role           = VOICE_ROLE,
    voice_rate           = VOICE_RATE,
    volume               = VOLUME,
    pitch                = PITCH,
    voice_autorate       = VOICE_AUTORATE,
    align_sub_audio      = True,
)

print(f"Generating TTS with type={TTS_TYPE}, voice='{VOICE_ROLE}' …")
trk = DubbingSrt(cfg=tts_cfg, out_ext='wav')
trk.dubbing()
trk.align()
trk.task_done()

_wav_candidates = list(Path(cache_folder).glob("*.wav")) + list(Path(OUTPUT_DIR).glob(f"*dubbed*.wav"))
if _wav_candidates:
    import shutil
    shutil.copy(str(_wav_candidates[0]), DUBBED_WAV)
    print(f"Dubbed audio saved to: {DUBBED_WAV}")
elif tts_cfg.target_wav_output and Path(tts_cfg.target_wav_output).exists():
    import shutil
    shutil.copy(tts_cfg.target_wav_output, DUBBED_WAV)
    print(f"Dubbed audio saved to: {DUBBED_WAV}")
else:
    print("WARNING: Dubbed audio not found. Check logs.")

## Cell 7 – Full pipeline: Video Translation (VTV)

Run this cell to execute the entire pipeline in one shot (ASR → Translate → TTS → Final MP4).
This is the recommended cell if you do not need to inspect intermediate results.

In [ ]:
import uuid, re
from pathlib import Path
from videotrans.task.trans_create import TransCreate
from videotrans.task.taskcfg import TaskCfgVTT
from videotrans.util import tools

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(INPUT_VIDEO).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

vtv_cfg = TaskCfgVTT(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = OUTPUT_DIR,
    # Language
    source_language_code = SOURCE_LANG,
    target_language_code = TARGET_LANG,
    # ASR
    recogn_type          = RECOGN_TYPE,
    model_name           = WHISPER_MODEL,
    is_cuda              = USE_CUDA,
    remove_noise         = REMOVE_NOISE,
    enable_diariz        = False,
    rephrase             = 0,
    fix_punc             = FIX_PUNC,
    # Translation
    translate_type       = TRANSLATE_TYPE,
    # TTS
    tts_type             = TTS_TYPE,
    voice_role           = VOICE_ROLE,
    voice_rate           = VOICE_RATE,
    volume               = VOLUME,
    pitch                = PITCH,
    voice_autorate       = VOICE_AUTORATE,
    video_autorate       = VIDEO_AUTORATE,
    align_sub_audio      = True,
    # Output
    subtitle_type        = SUBTITLE_TYPE,
    is_separate          = False,
    recogn2pass          = False,
    clear_cache          = True,
)

app_cfg.current_status = 'ing'
print("Starting full VTV pipeline …")

trk = TransCreate(cfg=vtv_cfg)
trk.prepare()
trk.recogn()
trk.trans()
trk.dubbing()
trk.align()
trk.assembling()
trk.task_done()

print(f"\nDone! Output files are in: {OUTPUT_DIR}")

## Cell 8 – Subtitle-only translation (STS)

Translate an existing SRT file without touching the video.

In [ ]:
import uuid
from pathlib import Path
from videotrans.task._translate_srt import TranslateSrt
from videotrans.task.taskcfg import TaskCfgSTS

# ── Override these for standalone SRT translation ──────────────────────────
STS_INPUT_SRT  = "/path/to/source.srt"   # source SRT
STS_OUTPUT_SRT = "/path/to/target.srt"   # where to save
STS_SOURCE     = SOURCE_LANG             # use vars from Cell 2, or override
STS_TARGET     = TARGET_LANG
STS_TRANS_TYPE = TRANSLATE_TYPE
# ───────────────────────────────────────────────────────────────────────────

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(STS_INPUT_SRT).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

sts_cfg = TaskCfgSTS(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = str(Path(STS_OUTPUT_SRT).parent),
    source_sub           = STS_INPUT_SRT,
    source_language_code = STS_SOURCE,
    target_language_code = STS_TARGET,
    translate_type       = STS_TRANS_TYPE,
)

trk = TranslateSrt(cfg=sts_cfg, out_format=0)
trk.trans()
trk.task_done()

if sts_cfg.target_sub and Path(sts_cfg.target_sub).exists():
    import shutil
    shutil.copy(sts_cfg.target_sub, STS_OUTPUT_SRT)
    print(f"Translated SRT saved to: {STS_OUTPUT_SRT}")
else:
    print(f"Translation done. Check: {cache_folder}")

## Cell 9 – Standalone TTS from an SRT file

Generate dubbed audio from any SRT without a video.

In [ ]:
import uuid
from pathlib import Path
from videotrans.task._dubbing import DubbingSrt
from videotrans.task.taskcfg import TaskCfgTTS

# ── Override for standalone TTS ────────────────────────────────────────────
TTS_INPUT_SRT  = "/path/to/your.srt"     # SRT to synthesise
TTS_OUTPUT_WAV = "/path/to/output.wav"   # output dubbed audio
TTS_TARGET_LANG = TARGET_LANG
TTS_VOICE = VOICE_ROLE
TTS_RATE  = VOICE_RATE
# ───────────────────────────────────────────────────────────────────────────

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(TTS_INPUT_SRT).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

tts_cfg = TaskCfgTTS(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = str(Path(TTS_OUTPUT_WAV).parent),
    target_sub           = TTS_INPUT_SRT,
    target_language_code = TTS_TARGET_LANG,
    tts_type             = TTS_TYPE,
    voice_role           = TTS_VOICE,
    voice_rate           = TTS_RATE,
    volume               = VOLUME,
    pitch                = PITCH,
    voice_autorate       = VOICE_AUTORATE,
    align_sub_audio      = True,
)

trk = DubbingSrt(cfg=tts_cfg, out_ext='wav')
trk.dubbing()
trk.align()
trk.task_done()

if tts_cfg.target_wav_output and Path(tts_cfg.target_wav_output).exists():
    import shutil
    shutil.copy(tts_cfg.target_wav_output, TTS_OUTPUT_WAV)
    print(f"Dubbed audio saved to: {TTS_OUTPUT_WAV}")
else:
    print(f"TTS done. Check: {cache_folder}")

## Cell 10 – Utilities

In [ ]:
# ── List available Edge-TTS voices for a given locale ──────────────────────
async def list_edge_tts_voices(locale_filter=""):
    """Print Edge-TTS voices, optionally filtered by locale (e.g. 'en-', 'it-')."""
    import edge_tts
    voices = await edge_tts.list_voices()
    for v in voices:
        name = v['ShortName']
        if locale_filter.lower() in name.lower():
            print(f"{name:40s}  gender={v['Gender']}  locale={v['Locale']}")

# Usage (uncomment and run):
# import asyncio
# asyncio.run(list_edge_tts_voices(locale_filter="en-"))

In [ ]:
# ── List supported ASR / Translation / TTS providers ──────────────────────
from videotrans import recognition, translator, tts

print("=== ASR Providers ===")
for i, name in enumerate(recognition.RECOGN_NAME_LIST):
    print(f"  {i:>2}: {name}")

print("\n=== Translation Providers ===")
for i, name in enumerate(translator.TRANSLASTE_NAME_LIST):
    print(f"  {i:>2}: {name}")

print("\n=== TTS Providers ===")
for i, name in enumerate(tts.TTS_NAME_LIST):
    print(f"  {i:>2}: {name}")

In [ ]:
# ── Inspect / preview an SRT file ─────────────────────────────────────────
import srt

def preview_srt(path, n=10):
    subs = list(srt.parse(Path(path).read_text(encoding='utf-8')))
    print(f"File: {path}")
    print(f"Total subtitles: {len(subs)}")
    for s in subs[:n]:
        print(f"  [{s.index:>4}] {s.start} --> {s.end}")
        print(f"         {s.content.strip()}")

# Usage:
# preview_srt(SOURCE_SRT)
# preview_srt(TARGET_SRT)

In [ ]:
# ── Check GPU / CUDA availability ─────────────────────────────────────────
try:
    import torch
    print(f"PyTorch : {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU : {torch.cuda.get_device_name(0)}")
        print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    print("PyTorch not installed.")

import subprocess, shutil
ffmpeg_path = shutil.which('ffmpeg')
print(f"\nffmpeg : {ffmpeg_path or 'NOT FOUND – install ffmpeg and add to PATH'}")